# 🎓 Student Career & Interest Analysis
**End-to-End Data Analysis Project**  
*Tech Stack: Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn*

---

## 🧹 Step 1 — Data Loading & Cleaning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

PALETTE = ['#4361EE','#F72585','#4CC9F0','#7209B7','#3A0CA3','#560BAD','#480CA8','#3F37C9']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})
print('Libraries imported ✅')

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df_raw = pd.read_csv('../data/student_data.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
print('Column Info:')
df_raw.info()
print('\nNull counts:')
print(df_raw.isnull().sum())
print(f'\nDuplicates: {df_raw.duplicated().sum()}')

In [ ]:
df = df_raw.copy()
df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]
df.drop_duplicates(inplace=True)
df.dropna(subset=['cgpa'], inplace=True)
df['favorite_subject'].fillna(df['favorite_subject'].mode()[0], inplace=True)
df['cgpa'] = df['cgpa'].astype(float).round(2)
print(f'Clean shape: {df.shape}')
df.describe()

## 📊 Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['cgpa'], bins=25, color='#4361EE', edgecolor='white', alpha=0.9)
ax.axvline(df['cgpa'].mean(), color='#F72585', lw=2, linestyle='--', label=f"Mean: {df['cgpa'].mean():.2f}")
ax.axvline(df['cgpa'].median(), color='#4CC9F0', lw=2, linestyle='--', label=f"Median: {df['cgpa'].median():.2f}")
ax.set_title('CGPA Distribution of Students')
ax.set_xlabel('CGPA'); ax.set_ylabel('Number of Students')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
career_counts = df['career_goal'].value_counts()
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(career_counts.index, career_counts.values, color=PALETTE[:len(career_counts)], edgecolor='white')
ax.set_title('Career Goal Distribution'); ax.set_xlabel('Number of Students')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
subj_counts = df['favorite_subject'].value_counts()
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=subj_counts.index, y=subj_counts.values, palette='husl', ax=ax)
ax.set_title('Favourite Subject Distribution')
ax.set_xlabel('Subject'); ax.set_ylabel('Count')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

## 📈 Step 3 — Key Features

In [ ]:
# Feature 1: Career Goal vs CGPA
order = df.groupby('career_goal')['cgpa'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=df, x='career_goal', y='cgpa', order=order, palette='husl', width=0.6, ax=ax)
ax.set_title('Career Goal vs CGPA')
ax.set_xlabel('Career Goal'); ax.set_ylabel('CGPA')
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Feature 2: Branch vs Subject Heatmap
pivot = df.pivot_table(index='branch', columns='favorite_subject', aggfunc='size', fill_value=0)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Branch vs Favourite Subject Heatmap')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Feature 3: Top 3 Trends
print('🏆 Top 3 Trends Summary')
print('─' * 40)
print(f"Most popular career : {df['career_goal'].value_counts().idxmax()}")
print(f"Most preferred subject: {df['favorite_subject'].value_counts().idxmax()}")
branch_cgpa = df.groupby('branch')['cgpa'].mean()
print(f"Highest avg CGPA branch: {branch_cgpa.idxmax()} ({branch_cgpa.max():.2f})")

## 📊 Step 4 — Additional Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
branch_counts = df['branch'].value_counts()
axes[0].pie(branch_counts.values, labels=branch_counts.index, autopct='%1.1f%%',
            colors=['#4361EE','#F72585','#4CC9F0','#7209B7'],
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Branch-wise Student Count')
sns.violinplot(data=df, x='branch', y='cgpa', palette=PALETTE[:4], inner='box', ax=axes[1])
axes[1].set_title('CGPA Distribution Across Branches')
plt.tight_layout(); plt.show()

In [ ]:
hobby_counts = df['hobby'].value_counts()
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=hobby_counts.index, y=hobby_counts.values, palette=sns.color_palette('husl', len(hobby_counts)), ax=ax)
ax.set_title('Hobby / Interest Distribution')
ax.set_xlabel('Hobby'); ax.set_ylabel('Number of Students')
plt.tight_layout(); plt.show()

## Correlation Heatmap & ML Prediction

In [ ]:
le = LabelEncoder()
df_enc = df[['branch','favorite_subject','career_goal','hobby','gender','cgpa','year']].copy()
for col in ['branch','favorite_subject','career_goal','hobby','gender']:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

corr = df_enc.corr()
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Correlation Heatmap (All Features)')
plt.tight_layout(); plt.show()

In [ ]:
X = df_enc.drop('cgpa', axis=1)
y = df_enc['cgpa']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'R² Score : {r2_score(y_test, y_pred):.4f}')
print(f'MAE      : {mean_absolute_error(y_test, y_pred):.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred, alpha=0.6, color='#4361EE', edgecolors='white', s=60)
lims = [min(y_test.min(), y_pred.min())-0.3, max(y_test.max(), y_pred.max())+0.3]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual CGPA'); ax.set_ylabel('Predicted CGPA')
ax.set_title(f'Actual vs Predicted CGPA  |  R²={r2_score(y_test, y_pred):.3f}')
ax.legend(); plt.tight_layout(); plt.show()

## 🧠 Step 5 — Insights & Conclusion

| # | Insight |
|---|--------|
| 1 | **Full Stack Developer** is the most popular career goal across all branches |
| 2 | **AI&ML branch** has the highest average CGPA (8.14), followed by CSE |
| 3 | **Computer Vision Engineers** achieve the highest average CGPA among career paths |
| 4 | **DBMS** is the most preferred subject overall, popular in both IT and CSE branches |
| 5 | **Machine Learning** dominates as the top subject in AI&ML branch |
| 6 | Gender distribution is relatively balanced across all branches (~55:45) |
| 7 | CGPA shows roughly normal distribution with a mean around 7.75 |
| 8 | Coding is the most popular hobby among students — aligning with STEM interests |